<a href="https://colab.research.google.com/github/Amynwabu/54zeros/blob/main/MKU_Integration_Techniques_Unit2_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MK:U — Integration Techniques
### L4 Data Analyst Apprenticeship (ST0118) · Module 2: Data Preparation and Analysis

This notebook contains a runnable Python script for **every slide** of `Unit_2_Integration_Techniques_V3.pptx` (45 slides), in slide order. Each section starts with a markdown cell naming the slide, followed by one or more code cells you can run directly in Google Colab.

**How to use this notebook:**
1. Run the **Setup** cell first — it installs/imports everything used later.
2. Work top to bottom, slide by slide, in your session.
3. Code cells marked `# PBL` are the hands-on activities — try them yourself before checking the worked solution underneath.
4. Real Network Rail / NHS / ONS links are used throughout — see each cell's comments for live data sources.

**Anchor scenario (used throughout):** Network Rail Maintenance Intelligence Platform — combining track inspection (Excel), maintenance logs (SQL), and delay data (API) to answer: *"Which track sections show the highest correlation between maintenance gaps and service delays?"*


## Setup — run this first

In [ ]:
# Setup — run once at the start of your session
import pandas as pd
import numpy as np
import requests
import json
import io
from datetime import datetime, timedelta

# Display options for cleaner output in Colab
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("Setup complete. pandas:", pd.__version__)


---
## Slide 1 — Title: Integration Techniques
*Amaka Adiuku · Integration Techniques · Day 1*

In [ ]:
# Slide 1: Title slide — no code required.
# This notebook follows the deck slide-by-slide.
print("MK:U — Integration Techniques")
print("L4 Data Analyst Apprenticeship (ST0118)")
print("Day 1")


---
## Slide 2 — Intended Learning Outcomes

- **ILO1:** Implement complete data analysis workflows — from requirements gathering through to insight generation
- **ILO2:** Choose and apply appropriate analytical tools and techniques (Excel, SQL, Python)
- **ILO4:** Assess data integration approaches when combining multiple sources with varying quality

In [ ]:
# Slide 2: ILOs — represented as a simple reference dict you can print anytime
ILOs = {
    "ILO1": "Implement complete data analysis workflows — from requirements gathering through to insight generation",
    "ILO2": "Choose and apply appropriate analytical tools and techniques (Excel, SQL, Python)",
    "ILO4": "Assess data integration approaches when combining multiple sources with varying quality",
}
for code_, text in ILOs.items():
    print(f"{code_}: {text}")


---
## Slide 3 — KSBs (L4 Data Analyst, ST0118)

| KSB | Description | Level |
|---|---|---|
| K3 | Principles of the data life cycle and routine data analysis tasks | Apply |
| K5 | The differences between structured and unstructured data | Apply |
| K10 | Approaches to combining data from different sources | Apply |
| K11 | Approaches to organisational tools and methods for data analysis | Apply, Develop |
| S4 | Analyse data sets taking account of different data structures and database designs | Introduce |

In [ ]:
# Slide 3: KSBs as a small reference DataFrame — useful to keep visible while you work
ksb_df = pd.DataFrame([
    ["K3",  "Principles of the data life cycle and routine data analysis tasks", "Apply"],
    ["K5",  "The differences between structured and unstructured data",          "Apply"],
    ["K10", "Approaches to combining data from different sources",               "Apply"],
    ["K11", "Approaches to organisational tools and methods for data analysis",  "Apply, Develop"],
    ["S4",  "Analyse data sets taking account of different data structures and database designs", "Introduce"],
], columns=["KSB", "Description", "Level"])
ksb_df


---
## Slide 4 — Timetable for Day 1

In [ ]:
# Slide 4: Timetable — represented as a schedule DataFrame
timetable = pd.DataFrame([
    ["09:30-10:15", "Databases and database design"],
    ["10:15-10:50", "Session continues"],
    ["10:50-11:00", "Break"],
    ["11:00-12:30", "Session: ETL & Data Formats"],
    ["12:30-13:30", "Lunch"],
    ["13:30-14:15", "Session: Data Analysis Workflow"],
    ["14:15-15:00", "Session: API Integration"],
    ["15:00-15:10", "Break"],
    ["15:10-17:00", "Group work & case study"],
], columns=["Time", "Activity"])
timetable


---
## Slide 5 — Reference Books

In [ ]:
# Slide 5: Reference books — keep this list handy for further reading
reference_books = [
    "McKinney, W. (2022) Python for Data Analysis, 3rd ed. O'Reilly.",
    "VanderPlas, J. (2016) Python Data Science Handbook. O'Reilly.",
    "Kleppmann, M. (2017) Designing Data-Intensive Applications. O'Reilly.",
    "Geron, A. (2022) Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow, 3rd ed. O'Reilly.",
]
for b in reference_books:
    print("-", b)


---
## Slide 6 — PBL Scenario: Network Rail Maintenance Intelligence Platform

🚆 **The Brief:** You are part of the Data & Analytics team at Network Rail. Combine three siloed sources:
- Track inspection records (Excel, weekly)
- Asset maintenance logs (SQL Server, near real-time)
- Delay incident feed (REST API, National Rail open data platform)

**Business question:** *"Which track sections show the highest correlation between maintenance gaps and service delays?"*

**Deliverables:** ETL design diagram · Python notebook (this one!) · validated integrated dataset · 2 visualisations · 3-min walkthrough

Real data: https://www.networkrail.co.uk/who-we-are/transparency-and-ethics/transparency/open-data-feeds/

In [ ]:
# Slide 6: PBL Scenario setup — we'll build small synthetic versions of all 3 NR sources
# so the pipeline works identically whether you're offline or online.

# --- Source 1: Track inspection records (simulates the weekly Excel export) ---
np.random.seed(42)
section_ids_raw = ['ECM01', 'ECM-002', ' ecm03 ', 'ECM-004', 'ECM005']  # deliberately inconsistent, like real life!

inspection_df = pd.DataFrame({
    'section_id': section_ids_raw,
    'inspection_date': ['15/01/2026', '16/01/2026', '17/01/2026', '18/01/2026', '19/01/2026'],
    'condition_score': [78, 65, 91, 54, 88],
    'last_maintenance': ['01/11/2025', '15/09/2025', '20/12/2025', '01/08/2025', '10/12/2025'],
})
print("Source 1 — Track Inspection (Excel):")
inspection_df


---
## Slide 7 — What is data and how it relates to information and knowledge?

**Data** → raw facts (e.g. `45`) · **Information** → data in context (`45 minutes delay`) · **Knowledge** → information that informs a decision (`this section needs urgent maintenance`)

In [ ]:
# Slide 7: Data -> Information -> Knowledge, illustrated with the NR scenario

raw_data = 45  # just a number — this is DATA
information = f"{raw_data} minutes delay on section ECM-001"  # data + context = INFORMATION
print("DATA:       ", raw_data)
print("INFORMATION:", information)

# KNOWLEDGE requires comparison/interpretation against other information
average_delay = 18
if raw_data > average_delay * 2:
    knowledge = "ECM-001 delay is more than double the network average — escalate for review"
else:
    knowledge = "ECM-001 delay is within normal range"
print("KNOWLEDGE:  ", knowledge)


---
## Slide 8 — Data Formats

| Type | Examples | Tools/Workflow |
|---|---|---|
| **1. Structured** | Rows, columns, fixed schema | Relational databases, SQL, BI tools, data warehouses |
| **2. Semi-structured** | JSON, XML, tagged records | NoSQL databases, document stores, ETL/ELT pipelines |
| **3. Unstructured** | Free text, media, documents, audio, video | Object storage, search/indexing, ML/NLP pipelines |

💡 *Data structure affects storage, cleaning, tooling, and workflow design.*

In [ ]:
# Slide 8: Data Formats — one example of each, all loadable in Python

# 1. STRUCTURED — rows, columns, fixed schema
structured_example = pd.DataFrame({
    'section_id': ['ECM-001', 'ECM-002'],
    'delay_mins': [45, 12]
})
print("1. STRUCTURED (DataFrame):")
print(structured_example, "\n")

# 2. SEMI-STRUCTURED — JSON, tagged but flexible
semi_structured_example = {
    "section_id": "ECM-001",
    "delays": [{"date": "2026-01-15", "mins": 45}, {"date": "2026-01-16", "mins": 12}]
}
print("2. SEMI-STRUCTURED (JSON):")
print(json.dumps(semi_structured_example, indent=2), "\n")

# 3. UNSTRUCTURED — free text, no fixed schema
unstructured_example = "Engineer note: track found buckled near mile marker 45, urgent repair needed before next service."
print("3. UNSTRUCTURED (free text):")
print(unstructured_example)


---
## Slide 9 — Data Types
*Data type is the kind of value stored in a field or column.*

In [ ]:
# Slide 9: Data Types in Python / pandas

example_row = {
    'section_id': 'ECM-001',          # str (text)
    'delay_mins': 45,                  # int (whole number)
    'length_km': 12.7,                 # float (decimal)
    'is_overdue': True,                # bool (True/False)
    'inspection_date': '2026-01-15',   # str -> should become datetime
}

for field, value in example_row.items():
    print(f"{field:<18} value={value!r:<15} python_type={type(value).__name__}")

# Always confirm types with df.dtypes once data is in a DataFrame
df_check = pd.DataFrame([example_row])
print("\npandas dtypes:")
print(df_check.dtypes)


---
## Slide 10 — The Big Picture: Where Data Lives and How You Interact With It

Every data analyst task starts with 3 questions:
1. Where is the data stored?
2. What tool do I use to interact with it?
3. What insight is needed?

💡 *Python can read ALL three storage types with a single line of code — this is why it becomes your most versatile tool.*

In [ ]:
# Slide 10: One language, three storage types — illustrated (not all will run without live access)

# 1. FILE (CSV/Excel) — pd.read_csv() / pd.read_excel()
print("From a FILE:    pd.read_csv('file.csv')  or  pd.read_excel('file.xlsx')")

# 2. DATABASE (SQL) — pd.read_sql()
print("From a DATABASE: pd.read_sql('SELECT * FROM maintenance_log', connection)")

# 3. API (web service) — requests + pd.DataFrame()
print("From an API:     pd.DataFrame(requests.get(url).json())")

print("\nSame DataFrame object results every time — that consistency is the whole point.")


---
## Slide 11 — Where Data Lives in Real Organizations

**Broaden source awareness — 5 categories:**
1. **Files** (CSV, Excel, JSON, text)
2. **Relational Databases** (SQL Server, PostgreSQL, MySQL, Oracle)
3. **Warehouses/Lakes** (cloud warehouse, data lake, lakehouse)
4. **APIs** (REST, JSON, live service endpoints, web data)
5. **SaaS Exports** (CRM, ERP, HR systems, ticketing tools)

In [ ]:
# Slide 11: 5 categories of data source — a reference dict mapping category to Python tool

data_sources = {
    "1. Files":               "pandas: pd.read_csv(), pd.read_excel(), open()",
    "2. Relational Databases": "SQLAlchemy + pandas: pd.read_sql()",
    "3. Warehouses/Lakes":    "Cloud SDKs (e.g. boto3 for AWS S3) + pandas",
    "4. APIs":                "requests + pandas: pd.DataFrame(requests.get(url).json())",
    "5. SaaS Exports":        "Usually exported as CSV/Excel first, then pandas",
}
for category, tool in data_sources.items():
    print(f"{category:<26} -> {tool}")


---
## Slide 12 — Where Excel Starts to Break

Common breaking points: file size limits, manual repetition, version control chaos, no live data connections, error-prone copy/paste.

In [ ]:
# Slide 12: Where Excel starts to break — demonstrated with a manual-repetition cost calculation

manual_steps_per_week = 6          # filter, sort, pivot, copy, paste, format
minutes_per_step = 5
weeks_per_year = 48

annual_manual_minutes = manual_steps_per_week * minutes_per_step * weeks_per_year
annual_manual_hours = annual_manual_minutes / 60

print(f"Manual Excel process: {manual_steps_per_week} steps x {minutes_per_step} min x {weeks_per_year} weeks")
print(f"= {annual_manual_hours:.1f} hours per year spent on a task a script runs in seconds")

# A Python script runs the same steps in well under a second, every time, identically
print("\nThe same process in Python: ~0.01 seconds, identical result, every single run.")


---
## Slide 13 — Excel to Python Transition

| Excel | → | Python |
|---|---|---|
| Formula (built-in functions, cell formulas) | → | Code (Python logic + libraries) |
| Manual copy/paste (time-consuming, error-prone) | → | Repeatable script (automate once, reuse) |
| One workbook (single file/sheet) | → | Multiple sources (databases, APIs, files) |
| Dashboard output (visual summaries) | → | Workflow pipeline (end-to-end automation) |

💡 *Same analytical thinking, more scalable workflow design.*

In [ ]:
# Slide 13: Excel to Python — the same logic expressed as code, side by side

# Excel: =SUM(B2:B10)  ->  Python:
delays = [45, 12, 78, 23, 56, 19, 34, 67, 8]
total_delay = sum(delays)
print(f"Excel '=SUM(B2:B10)'  ==  Python sum(delays) = {total_delay}")

# Excel: =AVERAGE(B2:B10)  ->  Python:
average_delay = sum(delays) / len(delays)
print(f"Excel '=AVERAGE(...)' ==  Python sum(delays)/len(delays) = {average_delay:.1f}")

# Or, once in a DataFrame, even simpler:
delay_series = pd.Series(delays)
print(f"\npandas equivalent: delay_series.sum() = {delay_series.sum()}, .mean() = {delay_series.mean():.1f}")


---
## Slide 14 — Moving Beyond Excel

In [ ]:
# Slide 14: Moving Beyond Excel — combining 2 'files' that Excel would need separate tabs for

excel_tab_1 = pd.DataFrame({'section_id': ['ECM-001', 'ECM-002'], 'condition_score': [78, 65]})
excel_tab_2 = pd.DataFrame({'section_id': ['ECM-001', 'ECM-002'], 'delay_mins': [45, 12]})

# In Excel: VLOOKUP or manual copy-paste between tabs
# In Python: one line, no manual matching
combined = pd.merge(excel_tab_1, excel_tab_2, on='section_id')
print("Combined in one line with pd.merge() — no VLOOKUP needed:")
combined


---
## Slide 15 — Python: Variables and Data Type

In [ ]:
# Slide 15: Python variables and data types — the absolute basics

# Variables: a name pointing to a value (like naming a cell)
section_id = "ECM-001"      # str
delay_mins = 45             # int
length_km = 12.7            # float
is_overdue = True           # bool

# Check any variable's type with type()
for name, value in [("section_id", section_id), ("delay_mins", delay_mins),
                     ("length_km", length_km), ("is_overdue", is_overdue)]:
    print(f"{name} = {value!r:<12} -> {type(value).__name__}")

# Variables can be reassigned and recalculated
delay_per_km = delay_mins / length_km
print(f"\ndelay_per_km = delay_mins / length_km = {delay_per_km:.2f}")


---
## Slide 16 — Python Packages and Modules

- **Module**: a reusable `.py` file containing functions, variables or classes
- **Package**: a folder that groups related modules together
- **How it works**: Create a module file → Import the module → Use the function in your script

💡 *Modules and packages help you organise, reuse and manage Python code.*

In [ ]:
# Slide 16: Python Packages and Modules — demonstrated with a simple inline 'module' function
# (In a real project this would live in its own .py file and be imported)

def MKU_module(arg):
    """Example module function — in a real project this lives in MKU_module.py"""
    print(f"arg = {arg}")

# This simulates: import MKU_module  ->  MKU_module.MKU_module('...')
MKU_module('we are using a module')

# Real packages we use throughout this notebook:
import pandas as pd    # the data analysis package
import numpy as np     # numerical computing package
import requests        # the package for calling APIs
print("\npandas, numpy, and requests are all packages — each containing many modules and functions.")


---
## Slide 17 — Python as the Bridge from Excel Thinking

In [ ]:
# Slide 17: Python as the bridge from Excel thinking — the dictionary in code form

excel_to_python = {
    "AutoFilter":    "df[df['delay_mins'] > 30]",
    "Sort A-Z":      "df.sort_values('delay_mins')",
    "PivotTable":    "df.groupby('section_id')['delay_mins'].mean()",
    "VLOOKUP":       "pd.merge(df1, df2, on='section_id')",
    "Insert Chart":  "df.plot(kind='bar')",
}
for excel_action, python_code in excel_to_python.items():
    print(f"Excel: {excel_action:<14} ->  Python: {python_code}")


---
## Slide 18 — Reading Multiple File Type

- Read CSV: load data from CSV files
- Read Excel: import data from Excel workbooks
- Later extend to SQL: connect to databases using SQL
- Later extend to API: pull data from web services and APIs

🎯 *One environment, multiple sources. Read, combine, and prepare data for analysis — all with Python.*

In [ ]:
# Slide 18: Reading multiple file types — all produce the SAME object type (a DataFrame)

# Reading a CSV (using io.StringIO to simulate a file without needing one saved to disk)
csv_text = "section_id,delay_mins\nECM-001,45\nECM-002,12"
df_from_csv = pd.read_csv(io.StringIO(csv_text))
print("From CSV:")
print(df_from_csv, "\n")

# Reading an Excel file would use the same pattern:
# df_from_excel = pd.read_excel('delay_report.xlsx')
print("From Excel (same pattern): pd.read_excel('delay_report.xlsx')")

# Later: SQL  -> pd.read_sql('SELECT * FROM table', engine)
# Later: API  -> pd.DataFrame(requests.get(url).json())
print("From SQL (later):  pd.read_sql(query, engine)")
print("From API (later):  pd.DataFrame(requests.get(url).json())")


---
## Slide 19 — Python Environments for Data Analyst

Common environments: Jupyter Notebook, Google Colab (this one!), VS Code, JupyterLab. Colab runs entirely in your browser — no installation needed, and it's free.

In [ ]:
# Slide 19: Confirm your Python environment details — useful for troubleshooting

import sys
print("Python version:", sys.version)
print("Running in Google Colab:", 'google.colab' in sys.modules)

# Check key package versions
import pandas, numpy, requests
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("requests:", requests.__version__)


---
## Slide 20 — ETL (Extract, Transform & Load) Core Concept

In [ ]:
# Slide 20: ETL core concept — the three-stage pattern, shown as a simple function skeleton

def extract():
    """Stage 1: pull data from a source — no changes made yet."""
    pass

def transform(df):
    """Stage 2: clean, reshape, standardise — most of your time goes here."""
    pass

def load(df):
    """Stage 3: write the clean result somewhere useful."""
    pass

print("ETL = Extract -> Transform -> Load")
print("Always in this order. Never skip a stage.")


---
## Slide 21 — Why ETL (Extract, Transform & Load)

In [ ]:
# Slide 21: Why ETL exists — without it, here's what raw multi-source data looks like

# Three messy, inconsistent sources (notice the different ID formats and date formats)
source_a = pd.DataFrame({'id': ['ecm01', 'ECM-002'], 'score': [78, 65]})
source_b = pd.DataFrame({'ID': ['ECM-001', 'ecm002'], 'delay': [45, 12]})

print("Source A IDs:", source_a['id'].tolist())
print("Source B IDs:", source_b['ID'].tolist())
print("\nNotice: these refer to the SAME sections, but won't match in a merge as-is.")
print("This is exactly the problem ETL is designed to solve.")


---
## Slide 22 — Why ETL? The Problem It Solves: Four Data Silos, Four Problems

| Source | Format | Problem |
|---|---|---|
| 1. Track Inspection | Excel, weekly | Mixed date formats |
| 2. Maintenance Logs | SQL Server, live | Different ID format (e.g. ECM01) |
| 3. Delay Feed | API, JSON | Only 100 records returned at a time |
| 4. Finance Records | CSV, monthly | Costs in mixed currencies |

**ETL Pipeline Solves This** → one clean, integrated, analysis-ready dataset: consistent `section_id`, missing values handled with a documented strategy, one currency, one timezone, one schema, a clear audit trail.

In [ ]:
# Slide 22: The four data silos — set up exactly as they appear in the NR scenario

# 1. Track Inspection (Excel) — mixed date formats
track_inspection = pd.DataFrame({
    'section_id': ['ECM01', 'ECM-002', 'ecm03'],
    'date': ['15/01/2026', '2026-01-16', '17.01.2026'],   # 3 different date formats!
})

# 2. Maintenance Logs (SQL) — different ID format
maintenance_logs = pd.DataFrame({
    'ID': ['ECM-001', 'ECM002', 'ECM-003'],                # inconsistent again
    'last_service': ['2025-11-01', '2025-09-15', '2025-12-20'],
})

# 3. Delay Feed (API/JSON) — paginated, only 100 records at a time (simulated small here)
delay_feed = {"page": 1, "results": [
    {"section_id": "ECM-001", "delay_mins": 45},
    {"section_id": "ECM-002", "delay_mins": 12},
]}

# 4. Finance Records (CSV) — mixed currencies
finance_records = pd.DataFrame({
    'section_id': ['ECM-001', 'ECM-002'],
    'cost': ['£1,250.00', '$1,580.00'],   # mixed currency symbols, as text!
})

print("Four silos loaded — note the inconsistencies in IDs, dates, and currency formatting.")
print("\nTrack inspection:\n", track_inspection)
print("\nMaintenance logs:\n", maintenance_logs)
print("\nFinance records:\n", finance_records)


---
## Slide 23 — How Organisation Use Data: Extract

**Key question:** where is the data stored, and what is the safest repeatable way to access it?

In [ ]:
# Slide 23: EXTRACT stage — pulling from each of the 3 NR sources

# Extract from "Excel" (we already created inspection_df earlier — re-confirm it loads cleanly)
def extract_inspection():
    """Simulates pd.read_excel('track_inspection.xlsx')"""
    return pd.DataFrame({
        'section_id': ['ECM01', 'ECM-002', ' ecm03 '],
        'inspection_date': ['15/01/2026', '16/01/2026', '17/01/2026'],
        'condition_score': [78, 65, 91],
    })

# Extract from "SQL" (simulates pd.read_sql())
def extract_maintenance():
    """Simulates pd.read_sql('SELECT * FROM maintenance_log', engine)"""
    return pd.DataFrame({
        'ID': ['ECM-001', 'ECM002', 'ECM-003'],
        'last_service': ['2025-11-01', '2025-09-15', '2025-12-20'],
    })

# Extract from "API" (real pattern — would call requests.get() against a live endpoint)
def extract_delays():
    """Simulates requests.get(api_url).json() -> pd.DataFrame()"""
    return pd.DataFrame([
        {"section_id": "ECM-001", "delay_mins": 45},
        {"section_id": "ECM-002", "delay_mins": 12},
        {"section_id": "ECM-003", "delay_mins": 78},
    ])

df_inspection = extract_inspection()
df_maintenance = extract_maintenance()
df_delays = extract_delays()

print(f"Extracted: {len(df_inspection)} inspection rows, {len(df_maintenance)} maintenance rows, {len(df_delays)} delay rows")


---
## Slide 24 — How Organisation Use Data: Transform

**Key question:** what needs fixing, reshaping, combining or validating before the data can be trusted?

In [ ]:
# Slide 24: TRANSFORM stage — standardise IDs and dates before anything else

def standardise_id(series):
    """Make every section_id read the same way: 'ECM-001' format"""
    return (series.astype(str).str.strip().str.upper()
            .str.replace(r'([A-Z]+)0?(\d+)', r'\1-\2', regex=True))

df_inspection['section_id'] = standardise_id(df_inspection['section_id'])
df_maintenance['section_id'] = standardise_id(df_maintenance['ID'])
df_maintenance = df_maintenance.drop(columns=['ID'])

print("After standardisation:")
print("Inspection IDs:", df_inspection['section_id'].tolist())
print("Maintenance IDs:", df_maintenance['section_id'].tolist())
print("Delay IDs:", df_delays['section_id'].tolist())
print("\nAll three sources now use the SAME format — ready to merge.")


---
## Slide 25 — Data Transformation Strategies

*Transformation is the analyst craft — turning raw data into reliable, meaningful insight.*

**The Transformation Pipeline (6 steps):**
1. **Clean** — remove errors, duplicates and irrelevant data; handle missing values
2. **Standardise** — align formats, data types and naming conventions for consistency
3. **Join** — combine data from multiple sources using keys and business rules
4. **Validate** — check quality, completeness and accuracy against rules and expectations
5. **Derive Metrics** — create calculated fields and aggregations that add business meaning
6. **Handle Anomalies** — detect and treat outliers and exceptions appropriately

*Iterate and refine as new data and needs emerge.*

In [ ]:
# Slide 25: The 6-step Transformation Pipeline — applied to a deliberately messy example

# MESSY SOURCE DATA (matches the slide's example exactly — note the genuinely mixed formats)
messy = pd.DataFrame({
    'Name':   [' ALICE ', 'bob', 'Charlie', 'alice', 'Bob '],
    'Date':   ['01/02/25', '1-2-2025', '01 Feb 25', '2025/02/01', '02/01/25'],
    'Amount': ['£1,200.50', '1200.5', '1,200.50', '1200,50', '£1,200 .50'],
})
print("MESSY SOURCE DATA:")
print(messy, "\n")

# Step 1: CLEAN — remove whitespace, standardise case
df_clean = messy.copy()
df_clean['Name'] = df_clean['Name'].str.strip().str.title()

# Step 2: STANDARDISE — align date format and amount format
# Real-world dates rarely share ONE format — parse each value's own format explicitly
def parse_messy_date(value):
    """Try each known format in turn — this is what 'standardise' looks like in practice."""
    value = value.strip()
    known_formats = ['%d/%m/%y', '%d-%m-%Y', '%d %b %y', '%Y/%m/%d']
    for fmt in known_formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except ValueError:
            continue
    return pd.NaT  # couldn't parse — flag it rather than guess

df_clean['Date'] = df_clean['Date'].apply(parse_messy_date)

def parse_messy_amount(value):
    """Strip currency symbols and stray spaces, then fix comma-as-decimal vs comma-as-thousands."""
    value = str(value).replace('£', '').replace(' ', '').strip()
    if ',' in value and '.' in value:
        value = value.replace(',', '')          # comma = thousands separator, e.g. 1,200.50
    elif ',' in value:
        value = value.replace(',', '.')         # comma = decimal separator, e.g. 1200,50
    return pd.to_numeric(value, errors='coerce')

df_clean['Amount'] = df_clean['Amount'].apply(parse_messy_amount)

print("TRUSTED, ANALYSIS-READY DATA:")
print(df_clean)
assert df_clean['Date'].notnull().all(), "Some dates failed to parse — check known_formats"
assert df_clean['Amount'].notnull().all(), "Some amounts failed to parse — check parse_messy_amount"
print("\nAll rows successfully standardised — 0 nulls in Date or Amount.")

# Step 3 (JOIN), 4 (VALIDATE), 5 (DERIVE METRICS), 6 (HANDLE ANOMALIES) come next
# — we'll do JOIN and DERIVE METRICS later in this notebook once we merge all 3 NR sources.


---
## Slide 26 — PBL Activity: Data Transformation Decision-making

*Download data transformation datasets from Canvas.*

**Discussion question:** Which transformation creates the most business value, and which one carries the highest risk?

In [ ]:
# Slide 26: PBL — Data Transformation Decision-making
# PBL: Try this yourself before checking the discussion notes below.

# In your group, rank these 6 transformations by (a) business value and (b) risk if done wrong:
transformations = ["Clean", "Standardise", "Join", "Validate", "Derive Metrics", "Handle Anomalies"]

# Build your own ranking here:
my_value_ranking = []   # e.g. ["Join", "Clean", ...]
my_risk_ranking = []    # e.g. ["Handle Anomalies", "Join", ...]

print("Transformations to rank:", transformations)
print("\nDiscuss with your group, then fill in your rankings above.")

# --- Worked discussion notes (reveal after your group has discussed) ---
discussion_notes = {
    "Join":             "Highest business value (enables cross-source insight) AND highest risk (silent row loss if IDs don't match)",
    "Handle Anomalies":  "High risk if done wrong — removing a 'real' outlier event can hide a genuine safety issue",
    "Clean":            "Foundational — low risk alone, but everything downstream depends on getting it right",
}
print("\nWorked discussion notes:")
for k, v in discussion_notes.items():
    print(f"- {k}: {v}")


---
## Slide 27 — How Organisation Use Data: Load

In [ ]:
# Slide 27: LOAD stage — writing the clean result somewhere useful

def load_to_csv(df, filename):
    """Write the clean DataFrame to a versioned CSV file"""
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} rows to {filename}")

# We'll use this once we have our final merged dataset later in the notebook
print("load_to_csv() function defined — ready to use once we have a final dataset.")


---
## Slide 28 — What Happens During the Load Stage

In [ ]:
# Slide 28: What happens during Load — validate BEFORE you save

def validate_before_load(df):
    """Basic quality gate — run this before every load"""
    checks = {
        "Row count > 0": len(df) > 0,
        "No fully-empty rows": not df.isnull().all(axis=1).any(),
        "section_id is unique": df['section_id'].is_unique if 'section_id' in df.columns else None,
    }
    for check, passed in checks.items():
        status = "N/A (column not present)" if passed is None else ("PASS" if passed else "FAIL")
        print(f"[{status}] {check}")
    return all(v for v in checks.values() if v is not None)

# Test it on our cleaned data from Slide 25 (no section_id column here — that check will show N/A, correctly)
validate_before_load(df_clean)


---
## Slide 29 — Group Work: ELT with Network Rail Open Data Feeds

🔗 https://www.networkrail.co.uk/who-we-are/transparency-and-ethics/transparency/open-data-feeds/

**Task:** Download Table 1410 from the ORR station usage page.

In [ ]:
# Slide 29: Group Work — ELT with real Network Rail / ORR open data
# PBL: this cell shows the PATTERN for pulling a real public dataset.
# Run it as-is if you have internet access in your Colab session.

orr_url = "https://www.networkrail.co.uk/who-we-are/transparency-and-ethics/transparency/open-data-feeds/"
print(f"Network Rail Open Data Feeds: {orr_url}")
print("ORR Table 1410 (station usage) is published as an Excel/CSV download from data.gov.uk / ORR.")
print()

# Real-world pattern for downloading an ORR/Network Rail published file (adjust URL to the exact file link):
# import pandas as pd
# orr_table_url = "<exact .csv or .xlsx link from the ORR statistics page>"
# df_orr = pd.read_csv(orr_table_url)   # or pd.read_excel(orr_table_url)
# df_orr.head()

print("Pattern: pd.read_csv(url) or pd.read_excel(url) — same as any local file, just with a web address.")
print("ELT = Extract, LOAD, then Transform — load raw data first, transform afterwards. Compare to ETL from Slide 20-22.")


---
## Slide 30 — Data Process Workflow

In [ ]:
# Slide 30: Data Process Workflow — the pipeline so far, as one callable sequence

def run_etl_pipeline():
    """The full ETL sequence we've built across Slides 20-28"""
    print("1. EXTRACT  -> pulling from Excel, SQL, API")
    insp = extract_inspection()
    maint = extract_maintenance()
    delays = extract_delays()

    print("2. TRANSFORM -> standardising IDs")
    insp['section_id'] = standardise_id(insp['section_id'])
    maint['section_id'] = standardise_id(maint['ID']) if 'ID' in maint.columns else maint['section_id']

    print("3. LOAD -> validating before saving")
    validate_before_load(insp)

    return insp, maint, delays

result = run_etl_pipeline()
print("\nPipeline run complete.")


---
## Slide 31 — Data Analysis Workflow

*Comparing 3 published Data Science lifecycle diagrams (Analytics Vidhya, DatabaseTown, Monalisha Kumari).*

Sources:
- https://www.analyticsvidhya.com/blog/2021/05/introduction-to-data-science-project-lifecycle/
- https://databasetown.com/6-steps-of-data-science-lifecycle/
- https://monalishakumari.medium.com/how-to-perform-exploratory-data-analysis-generic-steps-for-beginners-5811555234e5

⚠️ *Note: these are Data Science lifecycles, which include ML model building — beyond L4 scope. Our 8-stage workflow (next slides) is the correct L4-calibrated version.*

In [ ]:
# Slide 31: Comparing lifecycle models — Data Science vs L4 Data Analyst workflow

ds_lifecycle = ["Problem Definition", "Data Collection", "Data Preparation",
                "EDA", "Modelling/ML", "Visualisation", "Deployment"]

l4_workflow = ["Define", "Source", "Explore", "Clean",
               "Analyse", "Visualise", "Communicate", "Validate"]

print("Data Science lifecycle (broader, includes ML):")
print(" -> ".join(ds_lifecycle))
print()
print("L4 Data Analyst workflow (our focus — no ML model building at this level):")
print(" -> ".join(l4_workflow))


---
## Slide 32 — Key Stages of the Data Lifecycle

1. **Problem Definition** — stakeholders, requirements, governance issues, risk and mitigation
2. **Data Collection** — generated through various sources; acquired, mined, ingested
3. **Preparation** — cleaned, structured, validated, transformed into a usable format
4. **Data Management** — store securely and accessibly; organise, update, protect
5. **Analysis** — analysed to generate insights; modelling/feature extraction
6. **Visualisation** — aid business decisions; reports, dashboards; interpretation, prediction
7. **Archival/Destruction** — long-term storage or permanent destruction

In [ ]:
# Slide 32: Key Stages of the Data Lifecycle — as a structured reference

data_lifecycle = {
    "1. Problem Definition": ["Stakeholders", "Requirements", "Governance issues", "Risk and mitigation"],
    "2. Data Collection":    ["Generated through various sources", "Acquired, mined, ingested"],
    "3. Preparation":        ["Cleaned, structured, validated, transformed into usable format"],
    "4. Data Management":    ["Store securely and accessibly", "Organise, update, protect"],
    "5. Analysis":           ["Analysed to generate insights", "Modelling/Feature extraction"],
    "6. Visualisation":      ["Aid business decisions", "Reports, dashboards", "Interpretation, prediction"],
    "7. Archival/Destruction": ["Long-term storage", "Permanent destruction"],
}
for stage, details in data_lifecycle.items():
    print(f"\n{stage}")
    for d in details:
        print(f"   - {d}")


---
## Slide 33 — Data Analysis Workflow (L4 — the 8 stages we use)

In [ ]:
# Slide 33: The 8-Stage L4 Workflow — as a tracked checklist you can update as you work

workflow_stages = {
    "1. Define":      {"done": False, "note": "What question? Who needs the answer?"},
    "2. Source":      {"done": False, "note": "Where does the data come from?"},
    "3. Explore":     {"done": False, "note": "What does the data actually look like?"},
    "4. Clean":       {"done": False, "note": "Fix missing values, wrong types, formats"},
    "5. Analyse":     {"done": False, "note": "Apply the right method"},
    "6. Visualise":   {"done": False, "note": "Choose a chart that shows the finding"},
    "7. Communicate": {"done": False, "note": "Explain findings in plain language"},
    "8. Validate":    {"done": False, "note": "Does the answer make sense?"},
}

def show_workflow_status():
    for stage, info in workflow_stages.items():
        mark = "[x]" if info["done"] else "[ ]"
        print(f"{mark} {stage:<16} {info['note']}")

show_workflow_status()


---
## Slide 34 — Data Analysis: Asking the Right Question

A simple problem statement template:
- **We are looking at:** [topic]
- **For:** [stakeholder]
- **Because:** [reason/decision]
- **Success looks like:** [measurable outcome]
- **By:** [deadline]

In [ ]:
# Slide 34: Asking the right question — fill in the problem statement template

problem_statement = {
    "We are looking at":   "Track section delays on the ECML",
    "For":                  "The Route Asset Manager",
    "Because":              "They need to prioritise maintenance",
    "Success looks like":   "A ranked list of top 5 risk sections",
    "By":                   "Friday's team meeting",
}

print("PROBLEM STATEMENT")
print("-" * 40)
for k, v in problem_statement.items():
    print(f"{k}: {v}")

# Workflow stage 1 update
workflow_stages["1. Define"]["done"] = True


---
## Slide 35 — Data Analysis: Getting to Know Your Data

**Quick exploration checklist:** `df.shape` · `df.head()` · `df.dtypes` · `df.isnull().sum()` · `df.duplicated().sum()` · `df.describe()` · `df['col'].value_counts()`

**Good practice:** explore first · think before you clean · document changes · check results · focus on quality.

In [ ]:
# Slide 35: Getting to Know Your Data — the full exploration checklist, run on real data

# Build a slightly messier version of our NR data to practise on
explore_df = pd.DataFrame({
    'section_id': ['ECM-001', 'ECM-002', 'ECM-001', 'ECM-003', None],
    'delay_mins': [45, 12, 45, None, 78],
    'date': ['2026-01-15', '2026-01-16', '2026-01-15', '2026-01-17', '2026-01-18'],
})

print("df.shape:", explore_df.shape)
print("\ndf.head():")
print(explore_df.head())
print("\ndf.dtypes:")
print(explore_df.dtypes)
print("\ndf.isnull().sum():")
print(explore_df.isnull().sum())
print("\ndf.duplicated().sum():", explore_df.duplicated().sum())
print("\ndf.describe():")
print(explore_df.describe())

# COMMON CLEANING TASKS
explore_df['date'] = pd.to_datetime(explore_df['date'])
explore_df['delay_mins'] = explore_df['delay_mins'].fillna(0)  # think first: does NULL mean zero here?
explore_df = explore_df.drop_duplicates()
explore_df['section_id'] = explore_df['section_id'].str.strip().str.upper()
print(f"\n{len(explore_df)} rows after cleaning")

workflow_stages["3. Explore"]["done"] = True
workflow_stages["4. Clean"]["done"] = True


---
## Slide 36 — API Integration

In [ ]:
# Slide 36: Section title — API Integration
print("=" * 50)
print("SECTION: API INTEGRATION")
print("From files and databases, to live data sources")
print("=" * 50)


---
## Slide 37 — API Integration: Connecting to the Live Data

*Instead of waiting for someone to email you a weekly CSV file of train delays, an API allows your Python script to ask the central server: "Give me the delay status for all trains at London Euston right now," and the server hands it back instantly.*

In [ ]:
# Slide 37: A first, real API call — using the ONS Beta API (free, no key required)

ons_url = "https://api.beta.ons.gov.uk/v1/datasets"

try:
    response = requests.get(ons_url, timeout=10)
    print("Status code:", response.status_code)
    if response.status_code == 200:
        data = response.json()
        print("Top-level keys in the response:", list(data.keys()))
    else:
        print("Request did not succeed — check your internet connection or the URL.")
except requests.exceptions.RequestException as e:
    print("Could not reach the API right now:", e)
    print("(This is expected if Colab has no internet access in your current session.)")


---
## Slide 38 — Why APIs Matter

No more manual downloads · always up to date · combine many sources programmatically · same pandas skills apply once the data lands.

In [ ]:
# Slide 38: Why APIs matter — manual download vs API, time comparison

manual_download_minutes_per_week = 10
weeks_per_year = 48
manual_annual_hours = (manual_download_minutes_per_week * weeks_per_year) / 60

api_call_seconds = 2
api_annual_minutes = (api_call_seconds * weeks_per_year) / 60

print(f"Manual weekly download: {manual_download_minutes_per_week} min/week x {weeks_per_year} weeks = {manual_annual_hours:.1f} hours/year")
print(f"Automated API call:     {api_call_seconds} sec/week  x {weeks_per_year} weeks = {api_annual_minutes:.2f} minutes/year")
print(f"\nTime saved per year: ~{manual_annual_hours:.1f} hours")


---
## Slide 39 — What an API Is in Analyst Terms
An API is like asking a librarian (the server) for a specific book (data) by giving a clear request (URL + parameters) — they hand it back in a standard format (JSON) you can read.

In [ ]:
# Slide 39: What an API is, in analyst terms — the request/response pattern, explicitly

# The 5-step pattern, one more time, fully annotated
def explain_api_call():
    print("Step 1: ENDPOINT  - the web address for the data you want")
    print("        e.g. https://api.beta.ons.gov.uk/v1/datasets")
    print()
    print("Step 2: REQUEST   - requests.get(url) — like asking a question")
    print()
    print("Step 3: RESPONSE  - the server sends back a status code + data")
    print("        200 = success, 404 = not found, 401 = access denied")
    print()
    print("Step 4: PARSE     - response.json() turns the reply into a Python dict")
    print()
    print("Step 5: USE IT    - pd.DataFrame(data) — same object as Excel/SQL gave you")

explain_api_call()


---
## Slide 40 — API Specifics

5 things every API call involves:
1. **Endpoints** — the web address for the data you want, e.g. `https://api.example.com/customers`
2. **Parameters** — add information to your request, e.g. `?country=UK&year=2024`
3. **Authentication Keys** — most APIs need a key to check who you are, e.g. `x-api-key: YOUR_KEY_HERE`
4. **Rate Limits** — APIs limit how many requests you can make, e.g. `100 requests per minute`
5. **Response Handling** — APIs return data, usually in JSON format; always check for errors too

In [ ]:
# Slide 40: API Specifics — all 5 concepts demonstrated together

# 1. ENDPOINT
endpoint = "https://api.beta.ons.gov.uk/v1/datasets"

# 2. PARAMETERS — added to the request as a dict
params = {"limit": 5}   # e.g. ask for only 5 results

# 3. AUTHENTICATION KEYS — NOT needed for ONS, but here's the pattern for APIs that DO need one:
# headers = {"x-api-key": "YOUR_KEY_HERE"}   # NEVER hardcode a real key — use os.environ instead

# 4. RATE LIMITS — be polite, add a short pause between repeated calls
import time
def polite_get(url, params=None, pause=0.5):
    time.sleep(pause)  # respect rate limits
    return requests.get(url, params=params, timeout=10)

# 5. RESPONSE HANDLING — always check the status code before using the data
try:
    response = polite_get(endpoint, params=params)
    print("Status code:", response.status_code)
    if response.status_code == 200:
        data = response.json()
        print("Got a valid JSON response with keys:", list(data.keys())[:5])
    else:
        print(f"Request failed with status {response.status_code}")
except requests.exceptions.RequestException as e:
    print("Network error:", e)


---
## Slide 41 — Case Study

Applying API integration to a real Network Rail-style question using live, public UK data sources.

In [ ]:
# Slide 41: Case Study — pulling a real NESO (energy) dataset, no key required
# NESO = National Energy System Operator (formerly National Grid ESO, renamed Oct 2024)

neso_url = "https://api.neso.energy/api/3/action/datastore_search"
params = {"resource_id": "23021dfa-a3b8-4f62-93bc-b60186c53fa4", "limit": 5}

try:
    response = requests.get(neso_url, params=params, timeout=10)
    print("Status code:", response.status_code)
    if response.status_code == 200:
        result = response.json()
        records = result.get("result", {}).get("records", [])
        if records:
            df_neso = pd.DataFrame(records)
            print(df_neso.head())
        else:
            print("No records returned — the resource_id may need updating; check neso.energy/data-portal")
except requests.exceptions.RequestException as e:
    print("Could not reach NESO API:", e)

print("\nLive data portal: https://www.neso.energy/data-portal")


---
## Slide 42 — Case Study (continued)

Comparing API-sourced data against your cleaned NR dataset — the same techniques apply regardless of source.

In [ ]:
# Slide 42: Case Study continued — NHS England A&E statistics (open, no key required for published files)

nhs_url = "https://www.england.nhs.uk/statistics/statistical-work-areas/ae-waiting-times-and-activity/"
print(f"NHS England A&E statistics: {nhs_url}")
print("Published as downloadable XLS/CSV files (not a live REST API) — same pd.read_csv()/read_excel() pattern applies.")
print()

# Pattern for reading a real NHS published file once you have the exact URL:
# nhs_file_url = "<exact .csv URL from the NHS statistics page>"
# df_nhs = pd.read_csv(nhs_file_url)
# df_nhs.head()

print("Compare: ONS (API, Slide 37) vs NESO (API, Slide 41) vs NHS (published file, this slide)")
print("All three become a pandas DataFrame — the same skills apply across every source type.")


---
## Slide 43 — Group Work

Apply everything from today: combine an API source with your cleaned Excel/SQL data from earlier in the notebook.

In [ ]:
# Slide 43: Group Work — PBL
# PBL: In your group, complete this full mini-pipeline using sources from across this notebook.

# 1. Re-use your cleaned inspection data from Slide 35/23
group_work_inspection = df_inspection.copy() if "df_inspection" in dir() else extract_inspection()

# 2. Re-use your delay data from Slide 23
group_work_delays = df_delays.copy() if "df_delays" in dir() else extract_delays()

# 3. YOUR TASK: merge them and calculate the average delay per section
# Write your code below this line:

# --- starter scaffold ---
# merged = pd.merge(group_work_inspection, group_work_delays, on="section_id", how="left")
# summary = merged.groupby("section_id")["delay_mins"].mean()
# print(summary)

print("Group work scaffold ready — complete the merge and groupby above.")
print(f"Inspection rows available: {len(group_work_inspection)}")
print(f"Delay rows available: {len(group_work_delays)}")


---
## Slide 44 — PBL Scenario (recap)

🚆 **Network Rail — Maintenance Intelligence Platform** *(same brief as Slide 6 — now you have all the tools)*

Combine track inspection (Excel), maintenance logs (SQL), and delay feed (API) to answer:
*"Which track sections show the highest correlation between maintenance gaps and service delays?"*

**Deliverables:** ETL design diagram · Python notebook · validated integrated dataset · 2 visualisations · 3-min walkthrough

In [ ]:
# Slide 44: PBL Scenario — full pipeline, bringing every stage of this notebook together
# PBL: This is your capstone exercise. Run it, then extend it with your own analysis.

def full_nr_pipeline():
    print("STAGE 1: EXTRACT")
    insp = extract_inspection()
    maint = extract_maintenance()
    delays = extract_delays()
    print(f"  Extracted {len(insp)} inspection, {len(maint)} maintenance, {len(delays)} delay rows")

    print("STAGE 2: TRANSFORM")
    insp["section_id"] = standardise_id(insp["section_id"])
    maint["section_id"] = standardise_id(maint["ID"]) if "ID" in maint.columns else standardise_id(maint["section_id"])
    insp["inspection_date"] = pd.to_datetime(insp["inspection_date"], dayfirst=True, errors="coerce")

    print("STAGE 3: JOIN (merge all three sources)")
    merged = pd.merge(insp, maint, on="section_id", how="left")
    merged = pd.merge(merged, delays, on="section_id", how="left")
    print(f"  Merged dataset: {len(merged)} rows, {len(merged.columns)} columns")

    print("STAGE 4: VALIDATE")
    validate_before_load(merged)

    print("STAGE 5: DERIVE METRICS")
    merged["condition_score"] = pd.to_numeric(merged["condition_score"], errors="coerce")
    merged["delay_mins"] = pd.to_numeric(merged["delay_mins"], errors="coerce").fillna(0)
    # Derived feature: a simple risk score combining low condition + high delay
    merged["risk_score"] = (100 - merged["condition_score"]) + merged["delay_mins"]

    print("STAGE 6: LOAD")
    load_to_csv(merged, "nr_integrated_dataset.csv")

    return merged

final_dataset = full_nr_pipeline()
print("\nFinal integrated dataset:")
final_dataset.sort_values("risk_score", ascending=False)


### Slide 44 (continued) — Visualisation
*Deliverable: 2 insight visualisations*

In [ ]:
# Slide 44 continued: 2 insight visualisations for your PBL deliverable

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Visualisation 1: Risk score by section (bar chart)
final_dataset.sort_values("risk_score", ascending=False).plot(
    x="section_id", y="risk_score", kind="bar", ax=axes[0], legend=False, color="steelblue"
)
axes[0].set_title("Risk Score by Section")
axes[0].set_xlabel("Section ID")
axes[0].set_ylabel("Risk Score")

# Visualisation 2: Condition score vs delay (scatter)
axes[1].scatter(final_dataset["condition_score"], final_dataset["delay_mins"], s=80, color="darkorange")
axes[1].set_title("Condition Score vs Delay")
axes[1].set_xlabel("Condition Score")
axes[1].set_ylabel("Delay (mins)")

plt.tight_layout()
plt.show()

print("\n3-min walkthrough talking points:")
print("1. Which section has the highest risk_score, and why?")
print("2. Does a lower condition_score correlate with higher delay?")
print("3. What would you recommend the Route Asset Manager prioritise?")


---
## Slide 45 — Network Rail – New Measurement Train (NMT)

The NMT generates all three data types from Slide 8 on a single run: structured sensor readings, semi-structured JSON from onboard systems, and unstructured high-resolution video — a real-world illustration of why every technique in this notebook matters together.

In [ ]:
# Slide 45: Closing — Network Rail's New Measurement Train (NMT) ties the whole session together

nmt_data_types = {
    "Structured":      "Sensor readings (track geometry, gauge, cant) — rows and columns, fixed schema",
    "Semi-structured": "Onboard system JSON logs — tagged but flexible",
    "Unstructured":    "High-resolution track video — requires computer vision to extract meaning",
}

print("Network Rail's New Measurement Train (NMT) — one train, three data types:")
for dtype, description in nmt_data_types.items():
    print(f"\n{dtype}:")
    print(f"  {description}")

print("\n" + "=" * 60)
print("SESSION COMPLETE")
print("=" * 60)
print("""
You've covered, in order:
  - Data formats, types, and where data lives (Slides 7-19)
  - ETL: Extract, Transform, Load (Slides 20-30)
  - The 8-stage Data Analysis Workflow (Slides 31-35)
  - API Integration (Slides 36-43)
  - A full capstone PBL pipeline (Slide 44)

Your final integrated dataset is saved as "nr_integrated_dataset.csv".
Keep this notebook — it is portfolio evidence for K3, K5, K10, K11, S4.
""")
